In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")
print(type(model))
print("Device:", model.device)

c:\Users\taher\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 173.95it/s]


Model loaded successfully!
<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
Device: cuda:0


In [2]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [4]:
dataset  = load_dataset("FinGPT/fingpt-sentiment-train")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 76772
    })
})


In [5]:
example = dataset["train"][0]
print(example)

{'input': 'Teollisuuden Voima Oyj , the Finnish utility known as TVO , said it shortlisted Mitsubishi Heavy s EU-APWR model along with reactors from Areva , Toshiba Corp. , GE Hitachi Nuclear Energy and Korea Hydro & Nuclear Power Co. .', 'output': 'neutral', 'instruction': 'What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.'}


In [6]:
print("INSTRUCTION:")
print(example["instruction"])

print("\nINPUT:")
print(example["input"])

print("\nOUTPUT:")
print(example["output"])

INSTRUCTION:
What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.

INPUT:
Teollisuuden Voima Oyj , the Finnish utility known as TVO , said it shortlisted Mitsubishi Heavy s EU-APWR model along with reactors from Areva , Toshiba Corp. , GE Hitachi Nuclear Energy and Korea Hydro & Nuclear Power Co. .

OUTPUT:
neutral


In [7]:
messages = [
    {
        "role": "system",
        "content": "You are a financial sentiment classifier."
    },
    {
        "role": "user",
        "content": example["instruction"] + "\n\n" + example["input"]
    },
    {
        "role": "assistant",
        "content": example["output"]
    }
]

print(messages)

[{'role': 'system', 'content': 'You are a financial sentiment classifier.'}, {'role': 'user', 'content': 'What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.\n\nTeollisuuden Voima Oyj , the Finnish utility known as TVO , said it shortlisted Mitsubishi Heavy s EU-APWR model along with reactors from Areva , Toshiba Corp. , GE Hitachi Nuclear Energy and Korea Hydro & Nuclear Power Co. .'}, {'role': 'assistant', 'content': 'neutral'}]


In [8]:
from transformers import AutoTokenizer

formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text)

<|im_start|>system
You are a financial sentiment classifier.<|im_end|>
<|im_start|>user
What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.

Teollisuuden Voima Oyj , the Finnish utility known as TVO , said it shortlisted Mitsubishi Heavy s EU-APWR model along with reactors from Areva , Toshiba Corp. , GE Hitachi Nuclear Energy and Korea Hydro & Nuclear Power Co. .<|im_end|>
<|im_start|>assistant
neutral<|im_end|>



In [9]:
tokenized = tokenizer(
    formatted_text,
    return_tensors="pt"
)

print(tokenized["input_ids"].shape)

torch.Size([1, 99])


In [10]:
print("Number of tokens:", tokenized["input_ids"].shape[1])

Number of tokens: 99


In [11]:
print(tokenized["input_ids"][0])

tensor([151644,   8948,    198,   2610,    525,    264,   5896,  25975,  33365,
            13, 151645,    198, 151644,    872,    198,   3838,    374,    279,
         25975,    315,    419,   3669,     30,   5209,   5157,    458,   4226,
           504,    314,  42224,     14,  59568,     14,  30487,     92,    382,
          6639,    965,  62748,  60127,  28079,   7523,    506,     88,     73,
          1154,    279,  57853,  15549,   3881,    438,   5883,     46,   1154,
          1053,    432,   2805,  31240,  78553,  28101,    274,   9812,     12,
          2537,  17925,   1614,   3156,    448,  70473,    504,   8713,   6586,
          1154,  95441,  21863,     13,   1154,  29857,  15882,  30364,  37444,
         12354,    323,  11862,  39502,    609,  37444,   7420,   3539,     13,
           659, 151645,    198, 151644,  77091,    198,  59568, 151645,    198])


In [12]:
tokens = tokenizer.convert_ids_to_tokens(
    tokenized["input_ids"][0]
)

print(tokens)

['<|im_start|>', 'system', 'Ċ', 'You', 'Ġare', 'Ġa', 'Ġfinancial', 'Ġsentiment', 'Ġclassifier', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'user', 'Ċ', 'What', 'Ġis', 'Ġthe', 'Ġsentiment', 'Ġof', 'Ġthis', 'Ġnews', '?', 'ĠPlease', 'Ġchoose', 'Ġan', 'Ġanswer', 'Ġfrom', 'Ġ{', 'negative', '/', 'neutral', '/', 'positive', '}', '.ĊĊ', 'Te', 'oll', 'isu', 'uden', 'ĠVo', 'ima', 'ĠO', 'y', 'j', 'Ġ,', 'Ġthe', 'ĠFinnish', 'Ġutility', 'Ġknown', 'Ġas', 'ĠTV', 'O', 'Ġ,', 'Ġsaid', 'Ġit', 'Ġshort', 'listed', 'ĠMitsubishi', 'ĠHeavy', 'Ġs', 'ĠEU', '-', 'AP', 'WR', 'Ġmodel', 'Ġalong', 'Ġwith', 'Ġreactors', 'Ġfrom', 'ĠAre', 'va', 'Ġ,', 'ĠToshiba', 'ĠCorp', '.', 'Ġ,', 'ĠGE', 'ĠHit', 'achi', 'ĠNuclear', 'ĠEnergy', 'Ġand', 'ĠKorea', 'ĠHydro', 'Ġ&', 'ĠNuclear', 'ĠPower', 'ĠCo', '.', 'Ġ.', '<|im_end|>', 'Ċ', '<|im_start|>', 'assistant', 'Ċ', 'neutral', '<|im_end|>', 'Ċ']


In [13]:
%pip install -U peft

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print(lora_config)



LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'q_proj', 'v_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


In [15]:
lora_model = get_peft_model(
    model,
    lora_config
)

lora_model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [16]:
lora_model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [17]:
from datasets import load_dataset

dataset = load_dataset(
    "FinGPT/fingpt-sentiment-train",
    split="train"
)

small_dataset = dataset.select(range(500))

print(small_dataset)

Dataset({
    features: ['input', 'output', 'instruction'],
    num_rows: 500
})


In [18]:
def format_example(example):
    messages = [ 
        {
            "role": "system",
            "content": "You are a financial sentiment classifier."
        },
        {
            "role": "user",
            "content": example["instruction"] +"\n\n" + example["input"]
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted_text}

formatted_dataset = small_dataset.map(format_example)


In [19]:
print(formatted_dataset)

Dataset({
    features: ['input', 'output', 'instruction', 'text'],
    num_rows: 500
})


In [20]:
print(formatted_dataset[0]["text"])

<|im_start|>system
You are a financial sentiment classifier.<|im_end|>
<|im_start|>user
What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.

Teollisuuden Voima Oyj , the Finnish utility known as TVO , said it shortlisted Mitsubishi Heavy s EU-APWR model along with reactors from Areva , Toshiba Corp. , GE Hitachi Nuclear Energy and Korea Hydro & Nuclear Power Co. .<|im_end|>
<|im_start|>assistant
neutral<|im_end|>



In [21]:
def tokenize_for_training(example):
    prompt_messages = [
        {
            "role": "system", 
            "content": "You are a financial sentiment classifier."
        },
        {
            "role": "user",
            "content": example["instruction"] + "\n\n" + example["input"]
        }

    ]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True

    )
    full_tokens = tokenizer(
        example["text"],
        truncation=True,
        max_lenght=512
    )

    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length= 512
    )
    labels = full_tokens["input_ids"].copy()

    prompt_length = min(
        len(prompt_tokens["input_ids"]),
        len(labels)

    )

    labels[:prompt_length] = [-100] * prompt_length
    full_tokens["labels"] = labels
    return full_tokens

training_dataset = formatted_dataset.map(tokenize_for_training)
print(training_dataset)

Dataset({
    features: ['input', 'output', 'instruction', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 500
})


In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name())

#data collator
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=lora_model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"

)
print("Data collator ready")

CUDA available: True
GPU:  NVIDIA GeForce RTX 5060 Laptop GPU


In [26]:
from transformers import TrainingArguments, Trainer
lora_model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="./results/lora_financial_sentiment",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    report_to="none"
)
print("Training arguments ready")

Training arguments ready


In [28]:
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=training_dataset,
    data_collator=data_collator
)

print("Trainer created successfully")


Trainer created successfully


In [29]:
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU memory reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

GPU memory allocated: 2.89 GB
GPU memory reserved: 2.93 GB


In [31]:
training_result = trainer.train()

Step,Training Loss
10,0.164905
20,0.157359
30,0.202361
40,0.377079
50,0.224563
60,0.234454
70,0.248074
80,0.157012
90,0.245877
100,0.140892


In [32]:
trainer.save_model("./results/qwen_financial_lora")
tokenizer.save_pretrained("./results/qwen_financial_lora")

('./results/qwen_financial_lora\\tokenizer_config.json',
 './results/qwen_financial_lora\\chat_template.jinja',
 './results/qwen_financial_lora\\tokenizer.json')